# Module 12 lab — Prefill versus incremental decode

This notebook measures the two phases separately on one NVIDIA T4. Cache construction is kept outside the timed decode region.

In [ ]:
# Import the reusable benchmark functions from the companion Python file.
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

MODULE_DIR = Path.cwd()
if not (MODULE_DIR / 'prefill_decode_benchmarks.py').exists():
    MODULE_DIR = Path.cwd() / 'code' / 'module-12'
sys.path.insert(0, str(MODULE_DIR))

from prefill_decode_benchmarks import (
    benchmark_decode_contexts,
    benchmark_generation_lengths,
    benchmark_prefill_lengths,
    load_model,
    require_cuda,
)

device = require_cuda()
print('GPU:', torch.cuda.get_device_name(device))
print('PyTorch:', torch.__version__)

In [ ]:
# Load the same small FP16 model used in Modules 10 and 11.
from transformers import AutoTokenizer

model_id = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = load_model(model_id, device)
print('Model dtype:', next(model.parameters()).dtype)
print('Layers:', model.config.num_hidden_layers)
print('KV heads:', model.config.num_key_value_heads)

In [ ]:
# Measure full-prompt forward passes; each row includes prompt-token throughput.
prefill_rows = benchmark_prefill_lengths(
    model, device, [64, 128, 256, 512, 1024], repeats=8
)
prefill_df = pd.DataFrame(prefill_rows)
display(prefill_df)

In [ ]:
# Plot prefill latency and inspect whether larger GEMMs improve token throughput.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(prefill_df['prompt_tokens'], prefill_df['prefill_ms'], marker='o')
axes[0].set(xlabel='Prompt tokens', ylabel='GPU time (ms)', title='Prefill latency')
axes[1].plot(prefill_df['prompt_tokens'], prefill_df['prompt_tokens_per_sec'], marker='o')
axes[1].set(xlabel='Prompt tokens', ylabel='Prompt tokens/sec', title='Prefill throughput')
for axis in axes:
    axis.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Build each KV cache before timing one new-token decode step.
decode_rows = benchmark_decode_contexts(
    model, device, [64, 256, 512, 1024], repeats=8
)
decode_df = pd.DataFrame(decode_rows)
display(decode_df)
assert (decode_df['cache_length_after'] == decode_df['cache_length_before'] + 1).all()

In [ ]:
# Compare the latency curves, remembering that prefill and decode do different work.
plt.figure(figsize=(8, 4))
plt.plot(prefill_df['prompt_tokens'], prefill_df['prefill_ms'], marker='o', label='Prefill')
plt.plot(decode_df['context_tokens'], decode_df['decode_ms'], marker='o', label='Decode: one token')
plt.xlabel('Prompt / cached context length')
plt.ylabel('GPU execution time (ms)')
plt.title('T4 — Prefill versus incremental decode')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# Measure end-to-end generation to expose the growing contribution of decode.
generation_rows = benchmark_generation_lengths(
    model, tokenizer, device, [8, 16, 32, 64], repeats=3
)
generation_df = pd.DataFrame(generation_rows)
display(generation_df)

## Interpretation

Prefill processes many rows together and often exposes more efficient GEMMs. Decode processes one new row while reading the existing KV history. Use the measured latency, token throughput, and cache-length assertion together; do not infer a bottleneck from GPU utilization alone.